<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 4: Codificación con base a la media</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

## Idea principal

La codificación con base en la media sustituye cada categoría por el promedio de una variable numérica asociado con esa categoría. En este ejemplo, usaremos la temperatura mínima como variable numérica y el pronóstico como categoría; por tanto, cada pronóstico recibirá la temperatura mínima promedio observada en sus filas.

En problemas predictivos, si el promedio se calcula usando la variable objetivo, debe estimarse solo con los datos de entrenamiento y dentro de cada partición de validación. Calcularlo con todo el conjunto antes de dividirlo puede filtrar información del objetivo (*data leakage*) y producir métricas demasiado optimistas.

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('temps.txt', sep=r'\s+', header=None)  # Registros meteorológicos
data.columns = ['Temperatura máxima', 'Temperatura mínima', 'HDD', 'CDD', 'Rain']
d = ['Sábado', 'Domingo', 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes']
dias = [d[i % 7] for i in range(len(data))]
catego = ['Soleado', 'Mayormente soleado', 'Parcialmente nublado', 'Mayormente nublado', 'Nublado', 'Lluvioso']
prono = [catego[-1] if data['Rain'][i] > 0 else catego[int(round(1.2 * (data['HDD'][i] - data['CDD'][i]) + 2))] for i in range(len(data))]
df = pd.DataFrame(columns=['Días', 'Pronóstico', 'Temperatura máxima', 'Temperatura mínima'], index=range(len(data)))
df.iloc[:, 2:4] = data.iloc[:, 0:2]
df['Días'] = dias
df['Pronóstico'] = prono
df.head()

,Días,Pronóstico,Temperatura máxima,Temperatura mínima
0,Sábado,Parcialmente nublado,28.2,11.2
1,Domingo,Parcialmente nublado,29.1,10.7
2,Lunes,Mayormente nublado,26.1,10.4
3,Martes,Mayormente soleado,29.4,11.1
4,Miércoles,Mayormente soleado,29.8,11.1


In [ ]:
# Promedio de temperatura mínima para cada categoría de pronóstico


Pronóstico
Lluvioso                12.850000
Mayormente nublado      10.683333
Mayormente soleado      11.435714
Nublado                  9.500000
Parcialmente nublado    11.261111
Soleado                 11.800000
Name: Temperatura mínima, dtype: float64

,Días,Pronóstico,Temperatura máxima,Temperatura mínima,Promedio
0,Sábado,Parcialmente nublado,28.2,11.2,11.261111
1,Domingo,Parcialmente nublado,29.1,10.7,11.261111
2,Lunes,Mayormente nublado,26.1,10.4,10.683333
3,Martes,Mayormente soleado,29.4,11.1,11.435714
4,Miércoles,Mayormente soleado,29.8,11.1,11.435714


In [4]:
#Codificando a través de la codificación de Efectos
E=pd.get_dummies(df,columns=['Pronóstico'],drop_first=True,prefix='',prefix_sep='',)
E.head()

,Días,Temperatura máxima,Temperatura mínima,Promedio,Mayormente nublado,Mayormente soleado,Nublado,Parcialmente nublado,Soleado
0,Sábado,28.2,11.2,11.261111,0,0,0,1,0
1,Domingo,29.1,10.7,11.261111,0,0,0,1,0
2,Lunes,26.1,10.4,10.683333,1,0,0,0,0
3,Martes,29.4,11.1,11.435714,0,1,0,0,0
4,Miércoles,29.8,11.1,11.435714,0,1,0,0,0


In [5]:
CR,=set(df.Pronóstico.unique())-set(E.columns)
CR

'Lluvioso'

In [6]:
E.iloc[:,4:]=E.iloc[:,4:].astype('int')
idx_CR,=np.where(df.Pronóstico==CR)
E.iloc[idx_CR,4:]=-1
E.iloc[idx_CR]

,Días,Temperatura máxima,Temperatura mínima,Promedio,Mayormente nublado,Mayormente soleado,Nublado,Parcialmente nublado,Soleado
17,Martes,27.9,11.5,12.85,-1,-1,-1,-1,-1
44,Lunes,26.6,14.2,12.85,-1,-1,-1,-1,-1


In [7]:
# Regresión lineal
from sklearn.linear_model import LinearRegression

In [8]:
lin=LinearRegression()
lin.fit(E.iloc[:,4:],E['Temperatura mínima'])

LinearRegression()

In [9]:
E.columns[4:]

Index(['Mayormente nublado', 'Mayormente soleado', 'Nublado',
       'Parcialmente nublado', 'Soleado'],
      dtype='object')

In [10]:
lin.coef_

array([-0.57169312,  0.18068783, -1.75502646,  0.00608466,  0.54497354])

In [11]:
d_E={categorías:coeficientes for categorías,coeficientes in zip(E.columns[4:],lin.coef_)}
d_E

{'Mayormente nublado': -0.5716931216931218,
 'Mayormente soleado': 0.1806878306878308,
 'Nublado': -1.7550264550264554,
 'Parcialmente nublado': 0.00608465608465564,
 'Soleado': 0.5449735449735462}

In [12]:
d_E[CR]=lin.intercept_
d_E

{'Mayormente nublado': -0.5716931216931218,
 'Mayormente soleado': 0.1806878306878308,
 'Nublado': -1.7550264550264554,
 'Parcialmente nublado': 0.00608465608465564,
 'Soleado': 0.5449735449735462,
 'Lluvioso': 11.255026455026455}

In [13]:
df['P_Efectos']=df.Pronóstico.map(d_E)
df.head()

,Días,Pronóstico,Temperatura máxima,Temperatura mínima,Promedio,P_Efectos
0,Sábado,Parcialmente nublado,28.2,11.2,11.261111,0.006085
1,Domingo,Parcialmente nublado,29.1,10.7,11.261111,0.006085
2,Lunes,Mayormente nublado,26.1,10.4,10.683333,-0.571693
3,Martes,Mayormente soleado,29.4,11.1,11.435714,0.180688
4,Miércoles,Mayormente soleado,29.8,11.1,11.435714,0.180688


### ¿Qué considerar con alta cardinalidad?

One-hot crea una columna por categoría; su tamaño puede crecer mucho. La codificación por conteo/frecuencia y la codificación por media mantienen pocas columnas, pero pueden hacer que categorías distintas compartan el mismo valor o perder información de identidad.

La codificación por media puede sobreajustar, sobre todo cuando una categoría tiene pocas observaciones. En un problema supervisado, calcula los promedios solo dentro de los datos de entrenamiento y usa validación cruzada o suavizado para evitar fuga del objetivo. No hay una técnica mejor en todos los casos: compara alternativas dentro del mismo esquema de validación y considera el modelo, la cardinalidad y la interpretabilidad.

## Ejercicio aplicado: codificar suburbios para predecir precios reales

Usaremos `Data/melb_data.csv`, un conjunto real de ventas de propiedades en Melbourne. La variable categórica será `Suburb` y la variable objetivo será `Price`. Compararemos la **codificación con base en la media** (target/mean encoding) con one-hot, conteo y ordinal.

Para que la comparación sea directa, usaremos únicamente `Suburb` como predictor y el mismo modelo de regresión Ridge para todas las codificaciones. Evaluaremos con MAE y RMSE en dólares australianos (AUD). El objetivo es observar cómo cambian las representaciones; no se busca construir el mejor modelo de precios.

#### Cargar los datos y separar entrenamiento y prueba

La media de `Price` por suburbio utiliza la variable objetivo. Por eso, primero separamos los datos y ajustamos todas las transformaciones solo con entrenamiento. Para codificación por media, además generaremos valores **out-of-fold** para las filas de entrenamiento: cada fila se codifica con promedios calculados sin incluir su propio objetivo.

[kfold](https://medium.com/@luiscopete/como-implementar-un-k-fold-cross-validation-a-modelos-de-inteligencia-artificial-con-scikit-learn-eb0726c5ba55)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

datos = pd.read_csv('Data/melb_data.csv', usecols=['Suburb', 'Price'])
datos = datos.dropna(subset=['Suburb', 'Price']).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    datos[['Suburb']], datos['Price'], test_size=0.25, random_state=42
)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f'Observaciones: {len(datos):,}')
print(f'Suburbios distintos: {datos["Suburb"].nunique()}')
print(f'Entrenamiento: {len(X_train):,} | Prueba: {len(X_test):,}')
datos.head()

### Construir las cuatro codificaciones

La codificación por media usa un suavizado hacia la media global para que categorías con pocas observaciones no reciban promedios extremos. El parámetro `suavizado` controla cuánto se acercan esos promedios a la media global. Aquí usamos 20 como valor didáctico, no como hiperparámetro optimizado.

Las categorías de prueba que no aparecieron en entrenamiento reciben la media global en la codificación por media, cero en conteo y ordinal, y un vector de ceros en one-hot.

In [ ]:
def mapa_media_suavizada(categorias, objetivo, suavizado=20):
    tabla = pd.DataFrame({
        'categoria': np.asarray(categorias),
        'objetivo': np.asarray(objetivo)
    })
    media_global = float(tabla['objetivo'].mean())
    resumen = tabla.groupby('categoria')['objetivo'].agg(['mean', 'count'])
    mapa = (
        resumen['mean'] * resumen['count'] + media_global * suavizado
    ) / (resumen['count'] + suavizado)
    return mapa, media_global



In [ ]:
# One-hot: aprender el vocabulario solo en entrenamiento.
codificador_ohe = OneHotEncoder(handle_unknown='ignore')
X_train_ohe = codificador_ohe.fit_transform(X_train[['Suburb']])
X_test_ohe = codificador_ohe.transform(X_test[['Suburb']])


In [ ]:
# Conteo: número de apariciones de cada suburbio en entrenamiento.
mapa_conteos = X_train['Suburb'].value_counts()
X_train_conteo = X_train['Suburb'].map(mapa_conteos).fillna(0).to_numpy().reshape(-1, 1)
X_test_conteo = X_test['Suburb'].map(mapa_conteos).fillna(0).to_numpy().reshape(-1, 1)


In [ ]:
# Ordinal: orden alfabético arbitrario; no implica orden real de los suburbios.
mapa_ordinal = {categoria: i for i, categoria in enumerate(sorted(X_train['Suburb'].unique()))}
X_train_ordinal = X_train['Suburb'].map(mapa_ordinal).fillna(-1).to_numpy().reshape(-1, 1)
X_test_ordinal = X_test['Suburb'].map(mapa_ordinal).fillna(-1).to_numpy().reshape(-1, 1)


In [ ]:
# Media out-of-fold en entrenamiento para reducir el sobreajuste por fuga del objetivo.
X_train_media_oof = np.empty(len(X_train), dtype=float)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for indices_ajuste, indices_validacion in kf.split(X_train):
    mapa_fold, media_fold = mapa_media_suavizada(
        X_train.loc[indices_ajuste, 'Suburb'], y_train.iloc[indices_ajuste]
    )
    X_train_media_oof[indices_validacion] = (
        X_train.loc[indices_validacion, 'Suburb']
        .map(mapa_fold)
        .fillna(media_fold)
        .to_numpy()
    )


In [ ]:
# Para prueba se ajusta el mapa con todo el entrenamiento, nunca con y_test.
mapa_media, media_global = mapa_media_suavizada(X_train['Suburb'], y_train)
X_test_media = (
    X_test['Suburb'].map(mapa_media).fillna(media_global).to_numpy().reshape(-1, 1)
)
X_train_media = X_train_media_oof.reshape(-1, 1)

print(f'Media global de Price en entrenamiento: AUD {media_global:,.0f}')
print(f'Columnas one-hot generadas: {X_train_ohe.shape[1]}')

### 3. Comparar el desempeño en el mismo conjunto de prueba

Entrenaremos el mismo estimador Ridge para cada representación. MAE y RMSE más bajos indican menor error en esta partición; R² más alto indica que se explica más variación respecto a predecir una constante. Los resultados pueden variar con otra partición, y no sustituyen una validación cruzada completa.

In [ ]:
representaciones = {
    'Media suavizada (out-of-fold)': (X_train_media, X_test_media),
    'One-hot': (X_train_ohe, X_test_ohe),
    'Conteo': (X_train_conteo, X_test_conteo),
    'Ordinal': (X_train_ordinal, X_test_ordinal),
}

In [ ]:
resultados = []
for nombre, (X_train_codificado, X_test_codificado) in representaciones.items():
    modelo = Ridge(alpha=1.0)
    modelo.fit(X_train_codificado, y_train)
    predicciones = modelo.predict(X_test_codificado)
    resultados.append({
        'Codificación': nombre,
        'MAE (AUD)': mean_absolute_error(y_test, predicciones),
        'RMSE (AUD)': np.sqrt(mean_squared_error(y_test, predicciones)),
        'R²': r2_score(y_test, predicciones),
    })

comparacion = pd.DataFrame(resultados).sort_values('RMSE (AUD)')
comparacion.style.format({
    'MAE (AUD)': 'AUD {:,.0f}',
    'RMSE (AUD)': 'AUD {:,.0f}',
    'R²': '{:.3f}',
})

In [ ]:
# Ver la cantidad de datos que respalda algunos promedios por suburbio.
resumen_suburbios = (
    pd.DataFrame({'Suburb': X_train['Suburb'], 'Price': y_train})
    .groupby('Suburb')['Price']
    .agg(observaciones='size', precio_promedio='mean')
    .join(mapa_media.rename('media_codificada'))
)

print('Suburbios con más observaciones en entrenamiento:')
display(resumen_suburbios.sort_values('observaciones', ascending=False).head())
print('Suburbios con menos observaciones en entrenamiento:')
display(resumen_suburbios.sort_values('observaciones').head())

| Método | Ventajas | Desventajas y riesgos |
|---|---|---|
| **Media (target encoding)** | Una sola columna incluso con muchas categorías; incorpora una señal relacionada con el precio, puede resumir categorías de alta cardinalidad. | Usa el objetivo: puede filtrar información y sobreajustarse, sobre todo en categorías raras. Requiere separación correcta, validación out-of-fold y, a menudo, suavizado. Categorías distintas con medias similares se vuelven indistinguibles. |
| **One-hot** | No impone orden ni usa el objetivo para crear las columnas, conserva la identidad de cada categoría conocida. | Puede crear muchas columnas; las categorías no observadas quedan representadas por ceros y no tienen un coeficiente aprendido. |
| **Conteo/frecuencia** | Compacta y no usa el objetivo, por lo que es menos propensa a fuga del objetivo. | Describe cuán común es una categoría, no su relación con el precio; suburbios distintos pueden tener el mismo conteo. |
| **Ordinal** | Muy compacta y sencilla. | El orden numérico suele ser arbitrario para una variable nominal; un modelo puede interpretar distancias u orden que no existen. |

**Conclusión:** no hay una codificación universalmente superior. Compara las alternativas dentro del mismo esquema de validación y con el modelo que usarás. En este ejercicio el conteo de suburbio y el índice ordinal son representaciones distintas, pero ambas reducen toda la categoría a un solo número, la codificación por media añade información del precio y por eso exige precauciones adicionales.
